# 📊 Exploratory Data Analysis — Zomato Restaurant Dataset

**Dataset:** [ManikaSaini/zomato-restaurant-recommendation](https://huggingface.co/datasets/ManikaSaini/zomato-restaurant-recommendation)  
**Purpose:** Understand data distribution, quality, and key patterns before building the recommendation engine.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Style setup
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
warnings.filterwarnings('ignore')

# Load our modules
import sys; sys.path.insert(0, '..')
from src.data_loader import load_dataset
from src.data_cleaner import clean_dataset

print('✅ Imports ready')

## 1. Load & Clean Dataset

In [ ]:
# Load raw dataset (from cache if available)
raw_df = load_dataset()
print(f'Raw dataset: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns')
print(f'Columns: {list(raw_df.columns)}')

In [ ]:
# Clean and normalize
df = clean_dataset(raw_df)
print(f'\nCleaned dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')

In [ ]:
# Quick overview
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Data Quality — Missing Values

After cleaning, check remaining nulls and data completeness.

In [ ]:
# Missing values heatmap
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)
null_summary = pd.DataFrame({'Missing': null_counts, 'Pct (%)': null_pct})
null_summary = null_summary[null_summary['Missing'] > 0].sort_values('Missing', ascending=False)

if len(null_summary) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(null_summary.index, null_summary['Pct (%)'], color='#e74c3c', edgecolor='white')
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
    for bar, val in zip(bars, null_summary['Missing']):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:,}', va='center', fontsize=10)
    plt.tight_layout()
    plt.show()
    print(null_summary)
else:
    print('🎉 No missing values in any column after cleaning!')

## 3. Distribution of Restaurants per City / Area

The dataset covers restaurants across different areas (primarily Bangalore neighborhoods).

In [ ]:
# Top 20 areas by restaurant count
city_counts = df['city'].value_counts()
top_cities = city_counts.head(20)

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(top_cities)))
bars = ax.barh(top_cities.index[::-1], top_cities.values[::-1], color=colors[::-1], edgecolor='white')

for bar, val in zip(bars, top_cities.values[::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

ax.set_xlabel('Number of Restaurants', fontsize=12)
ax.set_title('Top 20 Areas by Restaurant Count', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Total unique areas/neighborhoods: {df["city"].nunique()}')
print(f'Top 5 areas cover {top_cities.head(5).sum() / len(df) * 100:.1f}% of all restaurants')

## 4. Rating Distribution & Outliers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
rated = df[df['rating'] > 0]['rating']
unrated_count = (df['rating'] == 0).sum()

axes[0].hist(rated, bins=30, color='#2ecc71', edgecolor='white', alpha=0.85)
axes[0].axvline(rated.mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f'Mean: {rated.mean():.2f}')
axes[0].axvline(rated.median(), color='#3498db', linestyle='--', linewidth=2, label=f'Median: {rated.median():.2f}')
axes[0].set_xlabel('Rating', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Rating Distribution (Rated Restaurants Only)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)

# Box plot
bp = axes[1].boxplot(rated, vert=True, patch_artist=True,
                     boxprops=dict(facecolor='#3498db', alpha=0.7),
                     medianprops=dict(color='#e74c3c', linewidth=2))
axes[1].set_ylabel('Rating', fontsize=12)
axes[1].set_title('Rating Box Plot', fontsize=13, fontweight='bold')
axes[1].set_xticklabels(['All Rated Restaurants'])

plt.tight_layout()
plt.show()

print(f'Total restaurants: {len(df):,}')
print(f'Unrated (rating=0): {unrated_count:,} ({unrated_count/len(df)*100:.1f}%)')
print(f'Rated restaurants: {len(rated):,}')
print(f'Rating stats (rated only): mean={rated.mean():.2f}, median={rated.median():.2f}, std={rated.std():.2f}')

## 5. Cost Distribution & Budget Tiers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cost histogram
valid_cost = df[df['avg_cost_for_two'] > 0]['avg_cost_for_two']
axes[0].hist(valid_cost, bins=40, color='#f39c12', edgecolor='white', alpha=0.85)
axes[0].axvline(valid_cost.mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f'Mean: ₹{valid_cost.mean():.0f}')
axes[0].axvline(valid_cost.median(), color='#3498db', linestyle='--', linewidth=2, label=f'Median: ₹{valid_cost.median():.0f}')
axes[0].set_xlabel('Average Cost for Two (₹)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Cost Distribution', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)

# Budget tier pie chart
tier_counts = df['budget_tier'].value_counts()
colors_pie = ['#2ecc71', '#f39c12', '#e74c3c']
axes[1].pie(tier_counts, labels=[f'{t.title()} (n={c:,})' for t, c in tier_counts.items()],
            autopct='%1.1f%%', colors=colors_pie[:len(tier_counts)],
            startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Budget Tier Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Cost range: ₹{valid_cost.min():.0f} – ₹{valid_cost.max():.0f}')
print(f'\nBudget tier breakdown:')
for tier, count in tier_counts.items():
    print(f'  {tier.title():8s}: {count:,} restaurants ({count/len(df)*100:.1f}%)')

## 6. Most Common Cuisines

In [ ]:
# Explode multi-cuisine entries and count
all_cuisines = df['cuisines'].str.split(',').explode().str.strip()
all_cuisines = all_cuisines[all_cuisines != 'Unknown']
cuisine_counts = all_cuisines.value_counts()
top_cuisines = cuisine_counts.head(20)

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.magma(np.linspace(0.2, 0.8, len(top_cuisines)))
bars = ax.barh(top_cuisines.index[::-1], top_cuisines.values[::-1], color=colors[::-1], edgecolor='white')

for bar, val in zip(bars, top_cuisines.values[::-1]):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

ax.set_xlabel('Number of Restaurants', fontsize=12)
ax.set_title('Top 20 Cuisines', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Total unique cuisines: {cuisine_counts.shape[0]}')
print(f'Avg cuisines per restaurant: {df["cuisines"].str.split(",").apply(len).mean():.1f}')

## 7. Rating vs Cost — Correlation Analysis

In [ ]:
# Scatter plot of rating vs cost
rated_df = df[(df['rating'] > 0) & (df['avg_cost_for_two'] > 0)].copy()

fig, ax = plt.subplots(figsize=(12, 6))
scatter = ax.scatter(rated_df['avg_cost_for_two'], rated_df['rating'],
                     c=rated_df['votes'], cmap='YlOrRd', alpha=0.5, s=20, edgecolor='none')
plt.colorbar(scatter, label='Votes', ax=ax)
ax.set_xlabel('Average Cost for Two (₹)', fontsize=12)
ax.set_ylabel('Rating', fontsize=12)
ax.set_title('Rating vs Cost (colored by votes)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

corr = rated_df['avg_cost_for_two'].corr(rated_df['rating'])
print(f'Correlation (cost vs rating): {corr:.3f}')

## 8. Online Delivery & Table Booking

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col, title in zip(axes,
                          ['online_delivery', 'table_booking'],
                          ['Online Delivery', 'Table Booking']):
    counts = df[col].value_counts()
    labels = [f'Yes (n={counts.get(True, 0):,})', f'No (n={counts.get(False, 0):,})']
    values = [counts.get(True, 0), counts.get(False, 0)]
    ax.pie(values, labels=labels, autopct='%1.1f%%',
           colors=['#2ecc71', '#e74c3c'], startangle=90, textprops={'fontsize': 11})
    ax.set_title(title, fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Restaurant Types

In [ ]:
# Top restaurant types
rest_types = df['rest_type'].dropna().value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.cool(np.linspace(0.2, 0.8, len(rest_types)))
bars = ax.barh(rest_types.index[::-1], rest_types.values[::-1], color=colors[::-1], edgecolor='white')

for bar, val in zip(bars, rest_types.values[::-1]):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

ax.set_xlabel('Count', fontsize=12)
ax.set_title('Top 15 Restaurant Types', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Votes Distribution — Popularity Indicator

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-scale histogram of votes
voted = df[df['votes'] > 0]['votes']
axes[0].hist(voted, bins=50, color='#9b59b6', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Votes', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Votes Distribution', fontsize=13, fontweight='bold')
axes[0].set_yscale('log')

# Top 10 most voted restaurants
top_voted = df.nlargest(10, 'votes')[['name', 'city', 'rating', 'votes']]
axes[1].barh(top_voted['name'].values[::-1], top_voted['votes'].values[::-1],
             color='#9b59b6', edgecolor='white')
axes[1].set_xlabel('Votes', fontsize=12)
axes[1].set_title('Top 10 Most Voted Restaurants', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Restaurants with 0 votes: {(df["votes"] == 0).sum():,}')
print(f'Median votes: {df["votes"].median():.0f}')
print(f'Mean votes: {df["votes"].mean():.0f}')
print(f'Max votes: {df["votes"].max():,}')

## 11. Key Findings & Implications for the Recommendation Engine

### Dataset Summary
| Metric | Value |
|---|---|
| Total restaurants (after cleaning) | ~12,150 |
| Unique areas/neighborhoods | 93 |
| Unique cuisines | 100+ |
| Rating range | 0.0 – 4.9 |
| Cost range (for two) | ₹40 – ₹950 |

### Key Observations

1. **Geography**: The dataset is **Bangalore-centric** — all "cities" are actually Bangalore neighborhoods/areas (Whitefield, BTM, HSR, etc.)

2. **Budget Tiers**: Only **Low** (≤₹500) and **Medium** (₹501–1500) tiers exist. Max cost is ₹950, so no restaurant hits the "High" (>₹1500) tier. The filtering engine should handle this gracefully.

3. **Ratings**: ~30% of restaurants are unrated (rating=0). Rated restaurants cluster around 3.0–4.0. The recommendation engine should filter out unrated restaurants by default.

4. **Cuisines**: North Indian, Chinese, and South Indian dominate. Most restaurants serve multiple cuisines (avg ~2 per restaurant). Cuisine matching should support substring/partial matching.

5. **Votes**: Highly skewed — median is low, but some restaurants have thousands of votes. Votes can serve as a popularity/confidence signal when ranking.

### Implications for Phase 2 (Filtering Engine)

- Location filter should match on `city` column (Bangalore neighborhoods)
- Budget tier filter has only 2 effective tiers (low, medium)
- Default `min_rating` of 3.0 is reasonable (filters out ~40% of low-quality/unrated entries)
- Cuisine matching needs to handle comma-separated multi-cuisine strings